# Index M&A Report Generation

This notebook demonstrates how to generate M&A (Mergers & Acquisitions) analysis reports for a portfolio of stocks.

## Workflow Overview

1. **Configure Environment**: Set up API keys and initialize services
2. **Define Tickers**: Specify the stocks to analyze
3. **Search M&A News**: Fetch M&A-related news using Bigdata.com API
4. **Extract Source Map**: Collect top sources for each company
5. **Extract M&A Data**: Identify companies as acquisition targets and extract deal specifics
6. **Generate Report**: Create table output with sources

## Output Format

The report produces a clean table showing:
- Target Company (with ticker)
- Acquirer
- Deal Value
- Status
- M&A Announcement Date

Plus a Sources section with links for each target company.


## Step 1: Environment Setup

Load environment variables and initialize required services.


In [53]:
import os
import sys
import json
import asyncio
from pathlib import Path
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify API keys are set
openai_key = os.getenv('OPENAI_API_KEY')
bigdata_key = os.getenv('BIGDATA_API_KEY')

print(f"OpenAI API Key: {'✓ Set' if openai_key else '✗ Missing'}")
print(f"Bigdata API Key: {'✓ Set' if bigdata_key else '✗ Missing'}")

if not openai_key or not bigdata_key:
    print("\n⚠️  Please create a .env file with your API keys:")
    print("   OPENAI_API_KEY=your_key_here")
    print("   BIGDATA_API_KEY=your_key_here")


OpenAI API Key: ✓ Set
Bigdata API Key: ✓ Set


## Step 2: Initialize Services

Set up the topic search service and report service.


In [54]:
from services.topic_search_service import TopicSearchService
from services.report_service import ReportService
from config.topics import MA_TOPICS

# Initialize services
topic_search_service = TopicSearchService(api_key=bigdata_key)
report_service = ReportService()

print(f"Topic Search Service: ✓ Initialized")
print(f"Report Service: ✓ Initialized (provider: {report_service.llm_service.provider_name})")
print(f"\nM&A Topics loaded: {len(MA_TOPICS)} topics")
for i, topic in enumerate(MA_TOPICS, 1):
    print(f"  {i}. {topic['topic_name']}: {topic['topic_text'][:60]}...")


Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x11065a510>


Topic Search Service: ✓ Initialized
Report Service: ✓ Initialized (provider: openai)

M&A Topics loaded: 6 topics
  1. M&A Activity: What material acquisition, merger, or divestiture activities...
  2. M&A Activity: Is {company} being acquired or in merger discussions with an...
  3. M&A Activity: What acquisition offers or takeover bids has {company} recei...
  4. M&A Activity: What strategic alternatives or sale processes is {company} e...
  5. M&A Activity: When did {company} announce or sign a definitive merger agre...
  6. M&A Activity: What are the key timeline milestones and closing dates for {...


## Step 3: Configure Analysis Parameters

Define the tickers to analyze and the lookback period.


In [55]:
import pandas as pd
# Configuration
#TICKERS = ["MSFT", "NFLX", "GOOGL", "AAPL"]  # Modify as needed

# Read entity IDs from US_500.csv (column is RP_ENTITY_ID, not Symbol)
df = pd.read_csv('US_5.csv')
print(f"CSV columns: {df.columns.tolist()}")

# Use RP_ENTITY_ID column
TICKERS = df['RP_ENTITY_ID'].tolist()

# print ticker length
print("Total entity IDs:", len(TICKERS))

LOOKBACK_DAYS = 90  # Number of days to search for news

# Calculate date range
end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=LOOKBACK_DAYS)

print(f"Analysis Configuration:")
print(f"  Tickers: {', '.join(TICKERS)}")
print(f"  Period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print(f"  Lookback: {LOOKBACK_DAYS} days")


CSV columns: ['RP_ENTITY_ID']
Total entity IDs: 25
Analysis Configuration:
  Tickers: DB06B0, C951A2, 8AM205, 94208D, ADF092, 873DB9, B2E6B5, 69345C, AD3C93, DA48E4, 8E82A6, EB5E78, D9B1C9, 4C37C5, 0BC29E, 35F4B5, 76F067, BB07E4, E68C3D, D8442A, A4BCDE, 859D62, C9881C, 64E346, 66ECFD
  Period: 2025-10-08 to 2026-01-06
  Lookback: 90 days


## Step 4: Search M&A News for Each Ticker

Fetch M&A-related news articles using the topic search service.


In [56]:
import asyncio
from typing import List, Dict
import time

async def search_ma_news_for_ticker(ticker: str, days: int, topic_search_service, custom_topics):
    """Search for M&A news for a single ticker."""
    print(f"\nSearching M&A news for {ticker}...")
    try:
        results = await topic_search_service.search_ticker(
            ticker=ticker,
            days=days,
            custom_topics=custom_topics
        )
        print(f"  ✓ {ticker}: Search completed")
        return ticker, results
                   
    except Exception as e:
        print(f"  ✗ {ticker}: Error - {e}")
        return ticker, None


async def search_ma_news_parallel(tickers: List[str], days: int, max_concurrent: int = 10):
    """Search for M&A news for all tickers in parallel."""
    
    # Create semaphore to limit concurrent tasks
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def process_with_semaphore(ticker: str):
        """Process a ticker with semaphore to limit concurrency."""
        async with semaphore:
            return await search_ma_news_for_ticker(
                ticker, 
                days, 
                topic_search_service, 
                MA_TOPICS
            )
    
    print(f"Searching M&A news for {len(tickers)} tickers with {max_concurrent} parallel workers...")
    
    # Create tasks for all tickers
    tasks = [process_with_semaphore(ticker) for ticker in tickers]
    
    # Run all tasks concurrently
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Build results dictionary
    all_results = {}
    for result in results:
        if isinstance(result, Exception):
            print(f"  ✗ Task failed with exception: {result}")
        elif isinstance(result, tuple) and len(result) == 2:
            ticker, ticker_results = result
            all_results[ticker] = ticker_results
    
    return all_results


# Run the search in parallel
start_time = time.time()

search_results = await search_ma_news_parallel(TICKERS, LOOKBACK_DAYS, max_concurrent=10)

elapsed_time = time.time() - start_time
print(f"\n{'='*50}")
print(f"Search complete for {len(TICKERS)} tickers")
print(f"Time taken: {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")
print(f"Average time per ticker: {elapsed_time/len(TICKERS):.1f} seconds" if TICKERS else "")
print(f"\nResults summary:")
successful = sum(1 for v in search_results.values() if v is not None)
failed = len(search_results) - successful
print(f"  ✓ Successful: {successful}")
print(f"  ✗ Failed: {failed}")

Searching M&A news for 25 tickers with 10 parallel workers...

Searching M&A news for DB06B0...

Searching M&A news for C951A2...

Searching M&A news for 8AM205...

Searching M&A news for 94208D...

Searching M&A news for ADF092...

Searching M&A news for 873DB9...

Searching M&A news for B2E6B5...

Searching M&A news for 69345C...

Searching M&A news for AD3C93...

Searching M&A news for DA48E4...
  ✓ B2E6B5: Search completed

Searching M&A news for 8E82A6...
  ✓ DB06B0: Search completed

Searching M&A news for EB5E78...
  ✓ ADF092: Search completed

Searching M&A news for D9B1C9...
  ✓ C951A2: Search completed

Searching M&A news for 4C37C5...
  ✓ 8AM205: Search completed

Searching M&A news for 0BC29E...
  ✓ 94208D: Search completed

Searching M&A news for 35F4B5...
  ✓ 873DB9: Search completed

Searching M&A news for 76F067...
  ✓ AD3C93: Search completed

Searching M&A news for BB07E4...
  ✓ DA48E4: Search completed

Searching M&A news for E68C3D...
  ✓ 69345C: Search completed

S

In [57]:
# print length of search_results
print(len(search_results))
# print each item topic_results for first item in search_results in json format
#print(json.dumps(search_results[TICKERS[0]].get('topic_results', []), indent=2))

    # Save to file
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
# Save search_results to json file so that it can be re-used
search_results_file = output_dir / f'search_results_{timestamp}.json'
with open(search_results_file, 'w') as f:
    json.dump(search_results, f, indent=2)

# print file saved at path
print(f"File saved at path: {f.name}")



25
File saved at path: output/search_results_20260106_163406.json


In [58]:
def extract_source_map(search_results: dict, top_n: int = 3) -> dict:
    """
    Extract top N sources (by relevance) for each company from search results.
    
    Args:
        search_results: Dictionary with ticker/entity_id as key and search data as value
        top_n: Number of top sources to extract (default: 3)
    
    Returns:
        Dictionary mapping company_name to source_map (source -> document_url)
    """
    company_sources = {}
    
    for ticker, data in search_results.items():
        if not data:
            continue
            
        company_name = data.get('company_name', ticker)
        topic_results = data.get('topic_results', [])
        
        # Collect all sources with relevance scores
        sources_with_relevance = []
        seen_sources = set()  # To avoid duplicates
        
        for result in topic_results:
            source = result.get('source', '')
            document_url = result.get('document_url')
            relevance = result.get('relevance', 0)
            
            # Create a unique key for source (source name + url)
            source_key = f"{source}|{document_url}"
            
            if source and source_key not in seen_sources:
                seen_sources.add(source_key)
                sources_with_relevance.append({
                    'source': source,
                    'document_url': document_url,
                    'relevance': relevance
                })
        
        # Sort by relevance (descending) and take top N
        sorted_sources = sorted(sources_with_relevance, key=lambda x: x['relevance'], reverse=True)[:top_n]
        
        # Create source_map: source -> document_url
        source_map = {}
        for item in sorted_sources:
            source_name = item['source']
            # If source name already exists, make it unique
            if source_name in source_map:
                source_name = f"{source_name} ({len([k for k in source_map if k.startswith(source_name)]) + 1})"
            source_map[source_name] = item['document_url']
        
        company_sources[company_name] = source_map
    
    return company_sources


# Extract source_map for each company
source_map_by_company = extract_source_map(search_results, top_n=3)

print(f"Source maps extracted for {len(source_map_by_company)} companies:")
for company, sources in source_map_by_company.items():
    print(f"\n  {company}:")
    for source, url in sources.items():
        url_display = url[:60] + '...' if url and len(url) > 60 else (url or 'N/A')
        print(f"    • {source}: {url_display}")


Source maps extracted for 25 companies:

  Electronic Arts Inc.:
    • Benzinga: https://www.benzinga.com/node/49588130?utm_campaign=partner_...
    • Edgar SEC Filings: https://www.sec.gov/Archives/edgar/data/712515/0001628280250...
    • Edgar SEC Filings (2): https://www.sec.gov/Archives/edgar/data/712515/0001140361250...

  Hologic Inc.:
    • Benzinga: https://www.benzinga.com/node/49572753?utm_campaign=partner_...
    • Edgar SEC Filings: https://www.sec.gov/Archives/edgar/data/859737/0000859737250...
    • Quartr Reports: https://files.quartr.com/reports/6271e-2025-12-23-09-46-43.p...

  Kenvue Inc.:
    • Edgar SEC Filings: https://www.sec.gov/Archives/edgar/data/1944048/000194404825...
    • Benzinga: https://www.benzinga.com/node/48674015?utm_campaign=partner_...
    • MT Newswires: N/A

  Dayforce Inc.:
    • The Fly: N/A
    • Edgar SEC Filings: https://www.sec.gov/Archives/edgar/data/1725057/000119312525...
    • Benzinga: https://www.benzinga.com/node/48878491?utm_campaig

## Step 5: Extract M&A Deal Data (Direct Extraction)

Extract structured M&A deal information directly from search results using the `brief_ma_specific` prompt. This identifies companies that are acquisition targets and extracts deal specifics in a table-ready format.


In [59]:
import asyncio
import yaml
import re
from typing import List, Dict
import pandas as pd

async def extract_ma_data_for_ticker(
    ticker: str, 
    results: dict, 
    llm_service,
    system_prompt: str,
    user_template: str,
    source_map: dict = None
) -> Dict:
    """Extract M&A deal data for a single ticker using brief_ma_specific prompt.
    
    Args:
        ticker: The ticker/entity_id
        results: Search results for this ticker
        llm_service: The LLM service instance
        system_prompt: System prompt for extraction
        user_template: User template for extraction
        source_map: Dictionary mapping source name to document_url (top 3 sources)
    
    Returns:
        Dictionary with M&A deal data or None if not a target
    """
    if not results or not results.get('topic_results'):
        print(f"  {ticker}: No results to process")
        return {"ticker": ticker, "is_target": False, "reason": "No search results"}
    
    company_name = results.get('company_name', ticker)
    print(f"  Extracting M&A data for {company_name}...")
    
    # Format topic results as context
    context = ""
    for topic_result in results.get('topic_results', []):
        context += f"Source: {topic_result.get('source', 'Unknown')}\n"
        context += f"Headline: {topic_result.get('headline', 'N/A')}\n"
        context += f"Content: {topic_result.get('text', topic_result.get('answer_chunk', ''))}\n"
        context += f"Document URL: {topic_result.get('document_url', 'N/A')}\n\n"
    
    # Render template
    current_datetime = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    user_prompt = user_template.replace('{{current_datetime}}', current_datetime)
    user_prompt = user_prompt.replace('{{company_name}}', company_name)
    user_prompt = user_prompt.replace('{{report}}', context)
    
    full_prompt = f"{system_prompt}\n\n{user_prompt}"
    
    try:
        response = await llm_service.generate_content_raw(prompt=full_prompt, model=None)
        
        # Parse JSON response
        # Extract JSON from response (handle markdown code blocks)
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)```', response)
        if json_match:
            json_str = json_match.group(1).strip()
        else:
            json_str = response.strip()
        
        ma_data = json.loads(json_str)
        ma_data['ticker'] = ticker
        ma_data['company_name'] = company_name
        
        # Add source_map
        if source_map:
            ma_data['source_map'] = source_map
        
        if ma_data.get('is_target', False):
            print(f"    ✓ {company_name}: IS acquisition target")
        else:
            print(f"    ○ {company_name}: Not a target - {ma_data.get('reason', 'Unknown')}")
        
        return ma_data
        
    except json.JSONDecodeError as e:
        print(f"    ✗ {company_name}: JSON parse error - {e}")
        return {"ticker": ticker, "company_name": company_name, "is_target": False, "reason": f"JSON parse error: {e}"}
    except Exception as e:
        print(f"    ✗ {company_name}: Error - {e}")
        return {"ticker": ticker, "company_name": company_name, "is_target": False, "reason": str(e)}


async def extract_ma_data_parallel(search_results, source_map_by_company: dict = None, max_concurrent: int = 10):
    """Extract M&A deal data for all tickers in parallel.
    
    Args:
        search_results: Dictionary of search results by ticker
        source_map_by_company: Dictionary mapping company_name to source_map
        max_concurrent: Maximum concurrent tasks
    
    Returns:
        Tuple of (ma_deals_list, sources_by_company)
    """
    # Load brief_ma_specific prompt
    with open('config/prompts.yaml', 'r', encoding='utf-8') as f:
        prompts = yaml.safe_load(f)
    
    prompt_config = prompts.get('brief_ma_specific', {})
    system_prompt = prompt_config.get('system_prompt', '')
    user_template = prompt_config.get('user_template', '')
    
    llm_service = report_service.llm_service
    
    # Create semaphore to limit concurrent tasks
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def process_with_semaphore(ticker: str, results: dict):
        """Process a ticker with semaphore to limit concurrency."""
        async with semaphore:
            company_name = results.get('company_name', ticker) if results else ticker
            source_map = source_map_by_company.get(company_name, {}) if source_map_by_company else {}
            return await extract_ma_data_for_ticker(
                ticker, results, llm_service, system_prompt, user_template, source_map
            )
    
    print(f"Extracting M&A data for {len(search_results)} tickers with {max_concurrent} parallel workers...")
    print("="*60)
    
    # Create tasks for all tickers
    tasks = [
        process_with_semaphore(ticker, results)
        for ticker, results in search_results.items()
    ]
    
    # Run all tasks concurrently
    results = await asyncio.gather(*tasks, return_exceptions=True)
    
    # Process results
    ma_deals = []
    sources_by_company = {}
    
    for result in results:
        if isinstance(result, Exception):
            print(f"  ✗ Task failed with exception: {result}")
        elif isinstance(result, dict):
            if result.get('is_target', False):
                ma_deals.append(result)
            # Always collect sources
            company_name = result.get('company_name', result.get('ticker', 'Unknown'))
            if result.get('source_map'):
                sources_by_company[company_name] = result.get('source_map')
    
    return ma_deals, sources_by_company


# Run M&A data extraction
import time
start_time = time.time()

ma_deals, sources_by_company = await extract_ma_data_parallel(search_results, source_map_by_company, max_concurrent=10)

elapsed_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"M&A extraction complete!")
print(f"  • Total tickers processed: {len(search_results)}")
print(f"  • Companies identified as targets: {len(ma_deals)}")
print(f"  • Time taken: {elapsed_time/60:.2f} minutes ({elapsed_time:.1f} seconds)")

Extracting M&A data for 25 tickers with 10 parallel workers...
  Extracting M&A data for Electronic Arts Inc....
  Extracting M&A data for Hologic Inc....
  Extracting M&A data for Kenvue Inc....
  Extracting M&A data for Dayforce Inc....
  Extracting M&A data for Warner Bros Discovery Inc....
  Extracting M&A data for Becton Dickinson & Co....
  Extracting M&A data for Atmos Energy Corp....
  Extracting M&A data for Advanced Micro Devices Inc....
  Extracting M&A data for AFLAC Inc....
  Extracting M&A data for Air Products & Chemicals Inc....
    ○ Air Products & Chemicals Inc.: Not a target - No M&A activity found
  Extracting M&A data for Albemarle Corp....
    ○ Advanced Micro Devices Inc.: Not a target - No M&A activity found
  Extracting M&A data for Skyworks Solutions Inc....
    ○ AFLAC Inc.: Not a target - No M&A activity found
  Extracting M&A data for American Electric Power Co. Inc....
    ○ Atmos Energy Corp.: Not a target - No M&A activity found
  Extracting M&A data for

In [60]:
# Build DataFrame from M&A deals
if ma_deals:
    # Create DataFrame with required columns
    df_deals = pd.DataFrame([
        {
            "Target Company": deal.get('target_company', deal.get('company_name', 'Unknown')),
            "Acquirer": deal.get('acquirer', 'Not disclosed'),
            "Deal Value": deal.get('deal_value', 'Not disclosed'),
            "Status": deal.get('status', 'Not disclosed'),
            "M&A Announcement Date": deal.get('announcement_date', 'Not disclosed')
        }
        for deal in ma_deals
    ])
    
    print(f"\n📊 M&A Activity Report - Companies as Acquisition Targets")
    print(f"   Period: {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
    print(f"   Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*100)
    
    # Display DataFrame
    display(df_deals)
else:
    print("No M&A activity found where listed companies are acquisition targets.")


📊 M&A Activity Report - Companies as Acquisition Targets
   Period: 2025-10-08 to 2026-01-06
   Generated: 2026-01-06 16:34:10


,Target Company,Acquirer,Deal Value,Status,M&A Announcement Date
0,Electronic Arts Inc. (EA),Not disclosed,$55 billion USD,Completed,Not disclosed
1,Hologic Inc. (HOLX),Blackstone and TPG,$18.3B USD,Announced,Not disclosed
2,Kenvue Inc. (KVUE),Kimberly-Clark Corporation,$48.7B USD,Announced,"November 3, 2025"
3,Dayforce Inc. (DAY),Thoma Bravo,$12.3B USD,Pending Regulatory Approval,Not disclosed
4,Warner Bros Discovery Inc. (WBD),"Netflix, Inc.",$82.7B USD,Pending Regulatory Approval,"December 05, 2025"
5,Becton Dickinson & Co. (BDX),Waters Corp,Not disclosed,Pending Regulatory Approval,Not disclosed
6,Skyworks Solutions Inc. (SWKS),Qorvo Inc.,$22B USD,Announced,Not disclosed
7,Analog Devices Inc. (ADI),ASE Technology Holding Co. Ltd.,Not disclosed,Not disclosed,Not disclosed


In [61]:
# Save M&A deals to CSV and JSON
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if ma_deals:
    # Save CSV
    csv_file = output_dir / f'ma_deals_{timestamp}.csv'
    df_deals.to_csv(csv_file, index=False)
    print(f"✓ CSV saved: {csv_file}")
    
    # Save JSON (includes full deal data)
    json_file = output_dir / f'ma_deals_{timestamp}.json'
    with open(json_file, 'w') as f:
        json.dump(ma_deals, f, indent=2)
    print(f"✓ JSON saved: {json_file}")

# Display Sources Section
print(f"\n{'='*100}")
print("📚 SOURCES BY COMPANY")
print("="*100)

# Get sources only for companies that are targets
target_companies = {deal.get('company_name', deal.get('ticker')) for deal in ma_deals} if ma_deals else set()

for company_name, sources in sources_by_company.items():
    if company_name in target_companies:
        print(f"\n{company_name}:")
        for source_name, url in sources.items():
            if url:
                print(f"  • {source_name}: {url[:80]}{'...' if len(str(url)) > 80 else ''}")
            else:
                print(f"  • {source_name}")

# Save sources to JSON
sources_file = output_dir / f'ma_sources_{timestamp}.json'
with open(sources_file, 'w') as f:
    # Only save sources for target companies
    filtered_sources = {k: v for k, v in sources_by_company.items() if k in target_companies}
    json.dump(filtered_sources, f, indent=2)
print(f"\n✓ Sources saved: {sources_file}")



✓ CSV saved: output/ma_deals_20260106_163410.csv
✓ JSON saved: output/ma_deals_20260106_163410.json

📚 SOURCES BY COMPANY

Electronic Arts Inc.:
  • Benzinga: https://www.benzinga.com/node/49588130?utm_campaign=partner_feed&utm_medium=feed...
  • Edgar SEC Filings: https://www.sec.gov/Archives/edgar/data/712515/000162828025047811/ea-20250930.ht...
  • Edgar SEC Filings (2): https://www.sec.gov/Archives/edgar/data/712515/000114036125046555/ef20061803_8k....

Hologic Inc.:
  • Benzinga: https://www.benzinga.com/node/49572753?utm_campaign=partner_feed&utm_medium=feed...
  • Edgar SEC Filings: https://www.sec.gov/Archives/edgar/data/859737/000085973725000072/holx-20250927....
  • Quartr Reports: https://files.quartr.com/reports/6271e-2025-12-23-09-46-43.pdf?ref=UmF2ZW5QYWNr

Kenvue Inc.:
  • Edgar SEC Filings: https://www.sec.gov/Archives/edgar/data/1944048/000194404825000198/kvue-20250928...
  • Benzinga: https://www.benzinga.com/node/48674015?utm_campaign=partner_feed&utm_medium=feed...


## Step 6: Generate Markdown Report

Convert the M&A deals DataFrame to a nicely formatted Markdown table with sources section.



In [62]:
from IPython.display import Markdown, display

def generate_markdown_report(df_deals: pd.DataFrame, sources_by_company: dict, start_date, end_date) -> str:
    """Generate a Markdown report with M&A deals table and sources.
    
    Args:
        df_deals: DataFrame with M&A deal information
        sources_by_company: Dictionary mapping company names to source maps
        start_date: Report period start date
        end_date: Report period end date
    
    Returns:
        Markdown formatted string
    """
    # Report header
    report = f"""# M&A Activity Report

**Period:** {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}  
**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  

---

## Companies as Acquisition Targets

"""
    
    # Generate table
    if df_deals is not None and not df_deals.empty:
        # Create markdown table header
        report += "| Target Company | Acquirer | Deal Value | Status | M&A Announcement Date |\n"
        report += "|---------------|----------|------------|--------|----------------------|\n"
        
        # Add rows
        for _, row in df_deals.iterrows():
            report += f"| {row['Target Company']} | {row['Acquirer']} | {row['Deal Value']} | {row['Status']} | {row['M&A Announcement Date']} |\n"
    else:
        report += "No M&A activity found where listed companies are acquisition targets during the specified period.\n"
    
    # Add sources section
    report += "\n---\n\n## Sources\n\n"
    
    # Get target companies from the deals
    target_companies = set()
    if df_deals is not None and not df_deals.empty:
        target_companies = set(df_deals['Target Company'].str.extract(r'^([^(]+)')[0].str.strip())
    
    for company_name, sources in sources_by_company.items():
        # Check if this company is in our target list (partial match)
        is_target = any(company_name in tc or tc in company_name for tc in target_companies)
        if is_target and sources:
            report += f"**{company_name}**\n"
            for source_name, url in sources.items():
                if url:
                    report += f"  - [{source_name}]({url})\n"
                else:
                    report += f"  - {source_name}\n"
            report += "\n"
    
        return report


# Generate the Markdown report
if ma_deals:
    ma_report = generate_markdown_report(df_deals, sources_by_company, start_date, end_date)
    print("✓ Markdown report generated!")
else:
    ma_report = generate_markdown_report(None, {}, start_date, end_date)
    print("⚠️ No M&A deals found - report shows empty table")

✓ Markdown report generated!


In [63]:
# Display the Markdown report
display(Markdown(ma_report))



# M&A Activity Report

**Period:** 2025-10-08 to 2026-01-06  
**Generated:** 2026-01-06 16:34:10  

---

## Companies as Acquisition Targets

| Target Company | Acquirer | Deal Value | Status | M&A Announcement Date |
|---------------|----------|------------|--------|----------------------|
| Electronic Arts Inc. (EA) | Not disclosed | $55 billion USD | Completed | Not disclosed |
| Hologic Inc. (HOLX) | Blackstone and TPG | $18.3B USD | Announced | Not disclosed |
| Kenvue Inc. (KVUE) | Kimberly-Clark Corporation | $48.7B USD | Announced | November 3, 2025 |
| Dayforce Inc. (DAY) | Thoma Bravo | $12.3B USD | Pending Regulatory Approval | Not disclosed |
| Warner Bros Discovery Inc. (WBD) | Netflix, Inc. | $82.7B USD | Pending Regulatory Approval | December 05, 2025 |
| Becton Dickinson & Co. (BDX) | Waters Corp | Not disclosed | Pending Regulatory Approval | Not disclosed |
| Skyworks Solutions Inc. (SWKS) | Qorvo Inc. | $22B USD | Announced | Not disclosed |
| Analog Devices Inc. (ADI) | ASE Technology Holding Co. Ltd. | Not disclosed | Not disclosed | Not disclosed |

---

## Sources

**Electronic Arts Inc.**
  - [Benzinga](https://www.benzinga.com/node/49588130?utm_campaign=partner_feed&utm_medium=feed&utm_source=ravenpack)
  - [Edgar SEC Filings](https://www.sec.gov/Archives/edgar/data/712515/000162828025047811/ea-20250930.htm)
  - [Edgar SEC Filings (2)](https://www.sec.gov/Archives/edgar/data/712515/000114036125046555/ef20061803_8k.htm)



## Step 7: Save Report

Save the M&A report to a Markdown file.

In [64]:
# Save the Markdown report to file
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

report_file = output_dir / f'ma_report_{timestamp}.md'
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(ma_report)

print(f"✓ Report saved to: {report_file}")


✓ Report saved to: output/ma_report_20260106_163410.md


## Summary

The workflow is complete. The following outputs have been generated:


In [65]:
# Print summary of outputs
print("📁 Output Files Generated:")
print(f"   • CSV:      output/ma_deals_*.csv")
print(f"   • JSON:     output/ma_deals_*.json")
print(f"   • Sources:  output/ma_sources_*.json")
print(f"   • Report:   output/ma_report_*.md")
print(f"\n📊 M&A Activity:")
print(f"   • Total tickers analyzed: {len(search_results)}")
print(f"   • Companies as acquisition targets: {len(ma_deals) if ma_deals else 0}")
print(f"\n✅ Workflow complete!")


📁 Output Files Generated:
   • CSV:      output/ma_deals_*.csv
   • JSON:     output/ma_deals_*.json
   • Sources:  output/ma_sources_*.json
   • Report:   output/ma_report_*.md

📊 M&A Activity:
   • Total tickers analyzed: 25
   • Companies as acquisition targets: 8

✅ Workflow complete!
